# CELL - 1 Load data 

In [0]:
import pandas as pd
import numpy as np

pdf = pd.read_parquet("/Volumes/workspace/default/retail_data/customer_features.parquet")
print(pdf.shape)
pdf.head()

(5878, 13)


,Customer ID,total_spent,avg_unit_price,num_invoices,num_line_items,first_purchase,last_purchase,num_unique_products,num_countries,recency_days,active_days,purchase_frequency,avg_basket_value
0,17764,1623.64,3.16,5,159,2009-12-03 14:28:00,2011-10-20 11:03:00,126,1,50,687,0.0073,324.73
1,15031,3156.58,2.83,14,405,2009-12-08 13:38:00,2011-12-05 14:19:00,237,1,4,728,0.0192,225.47
2,15785,5969.93,2.68,16,225,2009-12-15 16:28:00,2011-11-07 11:33:00,102,1,32,693,0.0231,373.12
3,18041,8752.38,2.05,38,960,2010-01-14 10:04:00,2011-11-28 11:39:00,464,1,11,684,0.0556,230.33
4,14344,3333.70,3.58,20,243,2010-03-23 15:47:00,2011-08-04 13:51:00,140,1,127,500,0.0400,166.69


# Cell 2 - Feature Engineering


In [0]:
fe = pdf.copy()

# 1. Log transform - xử lý long-tail
fe["log_total_spent"]         = np.log1p(fe["total_spent"])
fe["log_num_invoices"]        = np.log1p(fe["num_invoices"])
fe["log_num_unique_products"] = np.log1p(fe["num_unique_products"])
fe["log_avg_basket_value"]    = np.log1p(fe["avg_basket_value"])

# 2. Fill NaN purchase_frequency (active_days = 1 → div by 1, không NaN)
fe["purchase_frequency"] = fe["purchase_frequency"].fillna(0)

# 3. Drop datetime cols - không dùng để train
fe = fe.drop(columns=["first_purchase", "last_purchase"])

# 4. Summary
print("=== SHAPE ===")
print(fe.shape)

print("\n=== NULL CHECK ===")
print(fe.isnull().sum())

print("\n=== DESCRIBE ===")
fe.describe().round(2)

=== SHAPE ===
(5878, 15)

=== NULL CHECK ===
Customer ID                0
total_spent                0
avg_unit_price             0
num_invoices               0
num_line_items             0
num_unique_products        0
num_countries              0
recency_days               0
active_days                0
purchase_frequency         0
avg_basket_value           0
log_total_spent            0
log_num_invoices           0
log_num_unique_products    0
log_avg_basket_value       0
dtype: int64

=== DESCRIBE ===


,total_spent,avg_unit_price,num_invoices,num_line_items,num_unique_products,num_countries,recency_days,active_days,purchase_frequency,avg_basket_value,log_total_spent,log_num_invoices,log_num_unique_products,log_avg_basket_value
count,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00
mean,3018.62,8.58,6.29,137.04,81.99,1.00,200.87,274.39,0.32,391.73,6.84,1.55,3.76,5.64
std,14737.73,176.32,13.01,353.82,116.48,0.05,209.35,258.96,0.48,1215.07,1.39,0.81,1.22,0.74
min,2.95,0.15,1.00,1.00,1.00,1.00,0.00,1.00,0.00,2.95,1.37,0.69,0.69,1.37
25%,348.76,2.29,1.00,21.00,19.00,1.00,25.00,1.00,0.01,181.65,5.86,0.69,3.00,5.21
50%,898.92,2.96,3.00,53.00,45.00,1.00,95.00,222.00,0.03,285.08,6.80,1.39,3.83,5.66
75%,2307.09,3.85,7.00,142.00,103.00,1.00,379.00,513.00,1.00,420.57,7.74,2.08,4.64,6.04
max,608821.65,10953.50,398.00,12890.00,2550.00,2.00,738.00,739.00,3.00,84236.25,13.32,5.99,7.84,11.34


##  Feature Engineering Results

Từ 13 raw features → 15 features sau engineering (thêm 4 log-transformed).

### Transforms áp dụng:

**Log Transform** - xử lý long-tail distribution:
- `log_total_spent`, `log_num_invoices`, `log_num_unique_products`, `log_avg_basket_value`
- Lý do: các features này có std rất lớn so với mean
  (ví dụ total_spent: mean=3,018 nhưng std=14,737 và max=608,821)
  → log transform compress outliers, giúp model học tốt hơn

**Drop datetime cols:**
- `first_purchase`, `last_purchase` - đã được encode vào
  `recency_days` và `active_days`, không cần giữ raw timestamp

### Quan sát từ describe():

| Feature | Insight |
|---|---|
| `recency_days` mean=200 | Trung bình customer mua lần cuối cách đây 200 ngày |
| `purchase_frequency` mean=0.32 | Đa số mua < 1 lần/ngày, hoàn toàn hợp lý |
| `num_countries` max=2 | Hầu hết mua từ 1 quốc gia, ít reseller |
| `avg_basket_value` max=84,236 | Outlier rõ ràng - có thể là wholesale account |

In [0]:
# Save feature set
fe.to_parquet("/Volumes/workspace/default/retail_data/customer_features_engineered.parquet", index=False)
print("Saved!")

Saved!
